# Build a Hybrid Search Demo (Keyword + Embedding)

A runnable companion to the course project [*Build a Hybrid Search Demo (Keyword + Embedding)*](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/hybrid-search). This notebook runs **three retrieval methods on the same small corpus** — a from-scratch BM25-style keyword scorer, local embedding-based semantic search with `sentence-transformers`, and a hybrid that combines both — and shows, query by query, where each approach wins and loses.

**No API key, no signup, no `.env`.** The only downloads are the Python packages below and the small local embedding model (`all-MiniLM-L6-v2`, about 80MB, fetched on first run). These are the exact packages the local example project (`examples/hybrid-search/pyproject.toml`) declares, plus `pandas` for the pretty result tables — and the corpus and test queries are embedded right here in the notebook, so there's nothing to clone or upload. Works the same in Google Colab, Kaggle Notebooks, or Binder.


In [ ]:
!pip install -q sentence-transformers numpy pandas


## The corpus

The local companion example ships eleven short `.txt` passages under `data/corpus/`. We recreate the same passages here as Python strings so this notebook is fully self-contained.

The passages are deliberately written so keyword and semantic retrieval **disagree**:

- several passages use rare, distinctive vocabulary (an exact-match query will find them),
- and there are two *paraphrase* passages — one for the Neptune passage and one for the espresso passage — that say the same thing as their neighbor **without sharing any of its words** (an exact-match query will miss them entirely, while an embedding search finds them by meaning).

```text
neptune.txt              Neptune / ice giant / winds ...
telescope.txt            reflecting telescope / mirror ...
espresso.txt             espresso / crema ...
coldbrew.txt             cold brew / steeped / filtered ...
deepsea.txt              deep ocean / anglerfish / glowing lures ...
piano.txt                grand piano / sustain pedal ...
tomatoes.txt             tomatoes / ripening / temperatures ...
cycling.txt              bicycles / chainring / gears ...
sourdough.txt            sourdough / wild yeast / tang ...
paraphrase_neptune.txt   the distant ice giant located by calculation (no word "Neptune")
paraphrase_espresso.txt  the intense tiny serving brewed by pressure (no word "espresso")
```


In [ ]:
CORPUS = {
    "neptune.txt": (
        "Neptune is the eighth and farthest planet from the Sun, a deep-blue ice giant with "
        "the strongest recorded winds in the solar system. It was discovered in 1846 by "
        "astronomers who predicted its position mathematically from disturbances in the orbit "
        "of Uranus, long before any telescope had ever seen it. "
    ),
    "telescope.txt": (
        "A reflecting telescope gathers light with a curved primary mirror instead of a glass "
        "lens. Using a mirror avoids chromatic aberration, the colored fringing that afflicts "
        "refracting telescopes, and lets builders create much larger instruments for the same "
        "price. "
    ),
    "espresso.txt": (
        "An espresso is a concentrated coffee made by forcing hot water under nine bars of "
        "pressure through finely ground, tightly packed coffee. It forms the base of lattes, "
        "cappuccinos, and flat whites, and wears a layer of brown foam called crema on top. "
    ),
    "coldbrew.txt": (
        "Cold brew is coffee steeped in cold water for twelve to twenty-four hours and then "
        "filtered. Because the water never turns hot, cold brew tastes smoother and less "
        "acidic than coffee brewed with heat. "
    ),
    "deepsea.txt": (
        "The deep ocean below a thousand meters is cold, dark, and under crushing pressure. "
        "Its residents include anglerfish that dangle glowing lures and giant squids with eyes "
        "nearly as large as dinner plates. "
    ),
    "piano.txt": (
        "A grand piano produces sound when felt-covered hammers strike steel strings. Pressing "
        "the sustain pedal lifts the dampers so the strings keep vibrating after the key is "
        "released, letting notes blend into one another. "
    ),
    "tomatoes.txt": (
        "Tomatoes color up best when daytime temperatures stay between twenty and twenty-five "
        "degrees Celsius. Above thirty degrees, ripening slows sharply and the fruit stays "
        "greenish even though it is fully mature inside. "
    ),
    "cycling.txt": (
        "Bicycles transfer pedal power to the rear wheel through a chain and a set of gears. A "
        "small front chainring makes steep hills easier, while a large one trades torque for "
        "speed on flat ground. "
    ),
    "sourdough.txt": (
        "Sourdough bread rises through a slow fermentation of flour and water by wild yeast "
        "and lactobacillus bacteria. No commercial yeast is added, which is what gives the "
        "loaf its sour tang and open crumb. "
    ),
    "paraphrase_neptune.txt": (
        "The most distant ice giant in our solar system spins far beyond Saturn and Uranus. "
        "Astronomers located it by working out where a hidden planet must be to explain a "
        "strange wobble in Uranus's motion, then pointed their telescopes there and found it "
        "within a degree of the predicted spot. "
    ),
    "paraphrase_espresso.txt": (
        "Brewing an intense, tiny serving starts by pushing near-boiling water through very "
        "finely ground beans packed into a metal basket. Cafes steam milk on the side and pour "
        "the two together for their most popular hot drinks. "
    ),
}


documents = [{"id": doc_id, "text": text} for doc_id, text in sorted(CORPUS.items())]
print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  {doc['id']:<28} {doc['text'][:60]}...")


## Keyword search: a small BM25-style scorer

"Keyword search" here means a proper lexical scorer, not just `if term in text`. We implement a compact **BM25**-style function from scratch, because then you can read every line of the math:

- **idf (inverse document frequency):** `ln(1 + (N - df + 0.5) / (df + 0.5))` — a term that appears in only one document carries much more signal than one in every document.
- **term-frequency saturation:** the score each query term contributes grows with how often it appears in a document, but flattens out, so one document stuffed with a word doesn't blow everyone else away.
- **length normalization:** a 200-word document matching a term is stronger evidence than a 2,000-word document matching it once.

We also drop a small list of stopwords ("the", "with", "does", ...) from both sides — otherwise a query full of stopwords gives every document a tiny, nearly identical score and muddies the ranking.


In [ ]:
import math
import re
from collections import Counter

import numpy as np

_WORD_RE = re.compile(r"[a-z0-9']+")

STOPWORDS = frozenset("""
a an the and or but if then so of to in on for with at by from as is are
was were be been being do does did has have had it its this that these those
i you he she we they them their there here how why what which who whom when
where than too very can could will would should may might must not no yes
over under up down out off again all any both each few more most other some
such only own same about into through during before after above below
between because against rather
""".split())


def tokenize(text: str) -> list[str]:
    """Lowercases text, splits it into word tokens, and drops stopwords."""
    return [t for t in _WORD_RE.findall(text.lower()) if t not in STOPWORDS]


class KeywordScorer:
    """BM25-ish lexical scoring over the corpus, computed on the fly."""

    K1 = 1.5
    B = 0.75

    def __init__(self, documents):
        self.doc_ids = [doc["id"] for doc in documents]
        self.lengths = np.array([len(tokenize(doc["text"])) for doc in documents], dtype=float)
        self.avgdl = float(self.lengths.mean())

        self.doc_terms = []
        df = Counter()
        for doc in documents:
            terms = Counter(tokenize(doc["text"]))
            self.doc_terms.append(terms)
            df.update(terms.keys())

        n_docs = len(documents)
        self.idf = {
            term: math.log(1.0 + (n_docs - freq + 0.5) / (freq + 0.5))
            for term, freq in df.items()
        }

    def score(self, query: str) -> np.ndarray:
        """Raw BM25 score per document; higher is better, 0 = no shared vocabulary."""
        scores = np.zeros(len(self.doc_ids))
        for term in set(tokenize(query)):
            idf = self.idf.get(term, 0.0)
            if idf == 0.0:
                continue
            for i, terms in enumerate(self.doc_terms):
                tf = terms.get(term, 0)
                if tf:
                    denom = tf + self.K1 * (1 - self.B + self.B * self.lengths[i] / self.avgdl)
                    scores[i] += idf * (tf * (self.K1 + 1)) / denom
        return scores


scorer = KeywordScorer(documents)
print("Vocabulary size:", len(scorer.idf))


## Semantic search: local embeddings

An **embedding** is a list of numbers — a vector — that represents a piece of text's *meaning*, not its exact wording. `all-MiniLM-L6-v2` maps each passage to a point in 384-dimensional space, trained so that passages with similar meaning end up close together and unrelated passages far apart.

We embed every corpus passage once, then answer a query by embedding it the same way and ranking passages by **cosine similarity** — the cosine of the angle between two vectors. Since we normalize every vector to unit length, the cosine reduces to a plain dot product. Runs entirely locally, on your CPU, with no API key.


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode([doc["text"] for doc in documents], normalize_embeddings=True)
print("Embedding matrix shape:", embeddings.shape)  # (num_documents, 384)


def semantic_scores(query: str, embeddings: np.ndarray) -> np.ndarray:
    """Cosine similarity of the query against every document embedding."""
    query_vector = model.encode([query], normalize_embeddings=True)[0]
    return embeddings @ query_vector


## Hybrid search: combine the two scores

Keyword scores and cosine similarities live on completely different scales (BM25 scores can reach the tens; cosine similarities are roughly -1..1). So before combining, we **min-max normalize** each method's scores for the current query into the range [0, 1], then take a weighted average:

```text
hybrid = alpha * normalized_keyword + (1 - alpha) * normalized_semantic
```

With `alpha = 0.5` each method gets equal say. Lower `alpha` leans more semantic; higher leans more keyword.

Notice what happens on a paraphrase query: keyword finds *nothing*, every normalized keyword score is 0, and the hybrid collapses onto the semantic ranking — so hybrid automatically inherits whichever method actually has information for this query. That's the whole idea of hybrid search: **each method carries the queries the other one is blind to.**


In [ ]:
def minmax_normalize(scores: np.ndarray) -> np.ndarray:
    """Scale a score array to [0, 1] for one query. All-equal arrays map to zeros."""
    lo, hi = float(scores.min()), float(scores.max())
    if hi - lo < 1e-12:
        return np.zeros_like(scores)
    return (scores - lo) / (hi - lo)


def hybrid_scores(keyword: np.ndarray, semantic: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    return alpha * minmax_normalize(keyword) + (1 - alpha) * minmax_normalize(semantic)


def rank(documents, scores, top_k=3, drop_zeros=False):
    """Top-k documents for a score array, highest first."""
    order = np.argsort(scores)[::-1][:top_k]
    hits = []
    for i in order:
        if drop_zeros and scores[i] <= 0:
            continue
        hits.append({"id": documents[i]["id"], "score": float(scores[i]), "text": documents[i]["text"]})
    return hits


def retrieve_all(query, top_k=3, alpha=0.5):
    """Run all three methods for one query and return their top hits."""
    kw = scorer.score(query)
    sem = semantic_scores(query, embeddings)
    hy = hybrid_scores(kw, sem, alpha)
    return {
        "keyword": rank(documents, kw, top_k, drop_zeros=True),
        "semantic": rank(documents, sem, top_k),
        "hybrid": rank(documents, hy, top_k),
    }


## Compare all three on a few queries

Let's run the three methods on three representative queries and render each as a pandas table. Watch for the *pattern*, not just the winners:

- **exact-match query** (`Neptune planet winds`): all three find `neptune.txt` — keyword does it by vocabulary, embeddings by meaning, hybrid by both.
- **paraphrase query** (`grinding uphill against gravity`): keyword finds *nothing* (none of those words appear in the corpus), embeddings find `cycling.txt` by meaning, and hybrid keeps the semantic win.
- **tricky query** (`squeezing hot water through fine grounds for a quick cup`): keyword is led astray by the "hot water" overlap with the cold-brew passage; embeddings rank the espresso paraphrase first; hybrid — helped by the semantic side — does too.


In [ ]:
import pandas as pd


def show_query(query, expected=None, top_k=3):
    results = retrieve_all(query, top_k=top_k)
    rows = []
    for method, hits in results.items():
        for hit in hits:
            rows.append({
                "method": method,
                "score": round(hit["score"], 3),
                "document": hit["id"],
                "text (preview)": hit["text"][:64] + "..." if len(hit["text"]) > 64 else hit["text"],
            })
    print(f"Query: {query}" + (f"   |   expected: {expected}" if expected else ""))
    display(pd.DataFrame(rows).style.hide(axis="index"))
    print()


show_query("Neptune planet winds", expected="neptune.txt")
show_query("grinding uphill against gravity", expected="cycling.txt")
show_query("squeezing hot water through fine grounds for a quick cup", expected="paraphrase_espresso.txt")


In [ ]:
# The winners summary across a set of test queries
#
# One query doesn't prove anything, so the companion example also ships a
# bundled set of ten test queries (data/test_queries.json in the local
# project). Each query names the document that *should* rank first, and a note
# on whether it's an exact-match or a paraphrase query. We run all ten, and
# count how often each method got the expected document to rank 1.

TEST_QUERIES = [
    {"query": "Neptune planet winds", "expected": "neptune.txt", "kind": "keyword"},
    {"query": "an icy mystery world charted by pure calculation", "expected": "paraphrase_neptune.txt", "kind": "semantic"},
    {"query": "espresso crema", "expected": "espresso.txt", "kind": "keyword"},
    {"query": "squeezing hot water through fine grounds for a quick cup", "expected": "paraphrase_espresso.txt", "kind": "semantic"},
    {"query": "bicycle hill climbing gears", "expected": "cycling.txt", "kind": "keyword"},
    {"query": "grinding uphill against gravity", "expected": "cycling.txt", "kind": "semantic"},
    {"query": "why did my overnight dough get that bite", "expected": "sourdough.txt", "kind": "semantic"},
    {"query": "grand piano sustain pedal", "expected": "piano.txt", "kind": "keyword"},
    {"query": "a freezing abyss where animals make their own glow", "expected": "deepsea.txt", "kind": "semantic"},
    {"query": "reflecting telescope mirror aberration", "expected": "telescope.txt", "kind": "keyword"},
]

METHODS = ("keyword", "semantic", "hybrid")
wins = {method: 0 for method in METHODS}

rows = []
for item in TEST_QUERIES:
    results = retrieve_all(item["query"])
    winners = [m for m in METHODS if results[m] and results[m][0]["id"] == item["expected"]]
    for method in winners:
        wins[method] += 1
    rows.append({
        "query": item["query"][:44] + "..." if len(item["query"]) > 44 else item["query"],
        "kind": item["kind"],
        "expected": item["expected"],
        "won by": ", ".join(winners) if winners else "nobody",
    })

print("Per-query winners:")
display(pd.DataFrame(rows).style.hide(axis="index"))

print("Winners summary (how often each method got the expected doc to rank 1):")
display(pd.DataFrame({"method": METHODS, "wins": [wins[m] for m in METHODS]}).style.hide(axis="index"))


## Next steps

- **Tune the blend.** Re-run the winners summary with `alpha` closer to 0 (all-semantic) and 1 (all-keyword) and watch which queries change hands — the exact-match ones swing toward keyword, the paraphrase ones toward semantic.
- **Swap in your own corpus.** Replace `CORPUS` with a few passages on topics you actually care about, write a handful of test queries with the document each should match, and see whether the same pattern holds. On some corpora keyword will dominate; on others embeddings will. That's the honest lesson: **there is no "best" retriever in the abstract** — it depends on the query and the corpus, which is exactly why real search systems blend both.
- **See the full lesson** (with the step-by-step walkthrough of every cell above) at [`docs/projects/hybrid-search/index.md`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/hybrid-search) and the local, `uv`-based companion script at [`examples/hybrid-search/main.py`](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/examples/hybrid-search).
